# T-09 Prototype: BERTopic + HDBSCAN vs KMeans

**Goal:** Validate whether BERTopic with HDBSCAN beats the current KMeans+cosine baseline on silhouette score, before touching production `clusterer.py`.

**Acceptance (Lucas):** Do not merge until prototype silhouette ≥ KMeans baseline. `data_contracts.md` must be updated in the same PR on schema change.

**Datasets:** Revolut synthetic (8 files) + Notion synthetic (11 files) → ~300+ chunks across all 6 source types.

**Key questions:**
1. Does HDBSCAN cluster better than KMeans on our chunk embeddings?
2. How many chunks does HDBSCAN refuse to assign (noise / `topic = -1`)?
3. What fallback should we use for those noise chunks?
4. What new `clusterer.py` output shape does this require? (top_chunks + boundary_chunks for T-10)

In [21]:
"""Imports, paths, and reproducibility."""
import sys
from pathlib import Path

# Make pipeline/ importable from notebooks/
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Production pipeline modules — reuse, don't reimplement
from pipeline.chunker import chunk_text
from pipeline.embedder import embed_chunks

# Clustering libs
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from bertopic import BERTopic
from hdbscan import HDBSCAN
from umap import UMAP

# Reproducibility — UMAP is stochastic, fix the seed
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Data paths
DATA_DIR = PROJECT_ROOT / "data" / "synthetic"
REVOLUT_DIR = DATA_DIR / "revolut"
NOTION_DIR = DATA_DIR / "notion"

print(f"Project root: {PROJECT_ROOT}")
print(f"Revolut files: {len(list(REVOLUT_DIR.glob('*.txt')))}")
print(f"Notion files:  {len(list(NOTION_DIR.glob('*.txt')))}")

Project root: /Users/dmitriishumakher/Desktop/neuefische/capstoneProject/discovery-lens
Revolut files: 8
Notion files:  11


In [22]:
"""Load both synthetic datasets and chunk them through production chunker."""

# Map filename pattern → source_type (per CLAUDE.md enum)
def infer_source_type(filename: str) -> str:
    name = filename.lower()
    if "interview" in name:
        return "interview"
    if "review" in name:
        return "review"
    if "ticket" in name:
        return "ticket"
    if "usability" in name:
        return "usability"
    if "social" in name or "reddit" in name:
        return "social"
    if "internal" in name or "sales_notes" in name or "cs_notes" in name:
        return "internal"
    raise ValueError(f"Cannot infer source_type from: {filename}")


def load_corpus(directory: Path, product: str) -> list[dict]:
    """Run production chunker over a directory of .txt files."""
    all_chunks = []
    for filepath in sorted(directory.glob("*.txt")):
        raw_text = filepath.read_text(encoding="utf-8")
        source_type = infer_source_type(filepath.name)
        chunks = chunk_text(
            raw_text=raw_text,
            filename=filepath.name,
            source_type=source_type,
        )
        for c in chunks:
            c["product"] = product   # extra metadata for the prototype only
        all_chunks.extend(chunks)
    return all_chunks


revolut_chunks = load_corpus(REVOLUT_DIR, "revolut")
notion_chunks = load_corpus(NOTION_DIR, "notion")
chunks = revolut_chunks + notion_chunks

print(f"Revolut chunks: {len(revolut_chunks)}")
print(f"Notion chunks:  {len(notion_chunks)}")
print(f"Total chunks:   {len(chunks)}")

# Sanity — distribution by source_type
df_meta = pd.DataFrame(chunks)
print("\nChunks by source_type:")
print(df_meta["source_type"].value_counts())
print("\nChunks by product:")
print(df_meta["product"].value_counts())

[chunker:interview_revolut_01.txt] T-05 filtered 3 impoverished chunks (53 → 50)
[chunker:interview_revolut_02.txt] T-05 filtered 1 impoverished chunks (41 → 40)
[chunker:interview_revolut_03.txt] T-05 filtered 2 impoverished chunks (41 → 39)
[chunker:interview_revolut_04.txt] T-05 filtered 2 impoverished chunks (49 → 47)
[chunker:sales_notes_revolut.txt] T-05 filtered 1 impoverished chunks (26 → 25)
[chunker:usability_revolut_01.txt] T-05 filtered 12 impoverished chunks (36 → 24)
[chunker:usability_revolut_02.txt] T-05 filtered 4 impoverished chunks (28 → 24)
[chunker:usability_revolut_03.txt] T-05 filtered 9 impoverished chunks (32 → 23)
[chunker:internal_notion_cs_notes.txt] T-05 filtered 1 impoverished chunks (50 → 49)
[chunker:interview_notion_01.txt] T-05 filtered 2 impoverished chunks (38 → 36)
[chunker:interview_notion_02.txt] T-05 filtered 1 impoverished chunks (34 → 33)
[chunker:interview_notion_03.txt] T-05 filtered 3 impoverished chunks (36 → 33)
[chunker:interview_notion_0

In [23]:
"""Embed all chunks via production embedder (all-MiniLM-L6-v2)."""

texts = [c["text"] for c in chunks]
embeddings = embed_chunks(chunks)   # production function from pipeline/embedder.py

print(f"Embeddings shape: {embeddings.shape}")
print(f"Expected:         ({len(chunks)}, 384)")
assert embeddings.shape == (len(chunks), 384), "Shape mismatch — contract violation"
print("✓ Contract OK")

Embeddings shape: (799, 384)
Expected:         (799, 384)
✓ Contract OK


In [24]:
# Full source_type breakdown
with pd.option_context("display.max_rows", None):
    print("Chunks by source_type (full):")
    print(df_meta["source_type"].value_counts())
    print()
    print("Chunks by (product, source_type):")
    print(df_meta.groupby(["product", "source_type"]).size().sort_values(ascending=False))
    print()
    print(f"Total unique source_types: {df_meta['source_type'].nunique()}")

Chunks by source_type (full):
source_type
interview    382
usability    145
ticket        78
social        76
internal      74
review        44
Name: count, dtype: int64

Chunks by (product, source_type):
product  source_type
notion   interview      206
revolut  interview      176
notion   ticket          78
         social          76
         usability       74
revolut  usability       71
notion   internal        49
         review          44
revolut  internal        25
dtype: int64

Total unique source_types: 6


In [25]:
# What does production use for K?
import inspect
from pipeline import clusterer
print(inspect.getsource(clusterer))

from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np


def cluster(chunks: list[dict], embeddings: np.ndarray) -> list[dict]:
    n_chunks = len(chunks)
    k_values = range(3, min(8, n_chunks + 1)) # To work out the range of k values to try, we cap the maximum k at the number of chunks we actually have.
    
    best_k = 3          # fallback default in case something goes wrong
    best_score = -1     # silhouette scores range from -1 to 1, so -1 is worst possible

    for k in k_values:
        kmeans = KMeans(n_clusters=k, random_state=42, n_init=10) # n_init=10 means KMeans tries 10 different random starting points and keeps the best.

        labels = kmeans.fit_predict(embeddings) # fit_predict(): learns the k cluster centres from the embeddings and returns a label for each chunk: which cluster it belongs to
        score = silhouette_score(embeddings, labels) # silhouette_score measures how well-separated the clusters are. Higher is

In [26]:
"""KMeans baseline — exact reproduction of production clusterer.cluster()."""

# Production sweeps k from 3 to min(8, n_chunks+1) and picks the k with highest silhouette.
# It uses RAW embeddings (no normalization) and default euclidean silhouette.
# This must match exactly, or the comparison with HDBSCAN is unfair.

n_chunks = len(chunks)
k_values = range(3, min(8, n_chunks + 1) + 1)  # production: range(3, min(8, n_chunks + 1))

best_k = 3
best_score = -1.0
sweep_results = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(embeddings)
    score = silhouette_score(embeddings, labels)   # default = euclidean, matches production
    sweep_results.append({"k": k, "silhouette": score})
    if score > best_score:
        best_score = score
        best_k = k

sweep_df = pd.DataFrame(sweep_results)
print("KMeans k-sweep (production logic):")
print(sweep_df.to_string(index=False))

# Re-run with the winning k (production also does this — final_kmeans)
final_kmeans = KMeans(n_clusters=best_k, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = final_kmeans.fit_predict(embeddings)
kmeans_silhouette = silhouette_score(embeddings, kmeans_labels)

kmeans_sizes = pd.Series(kmeans_labels).value_counts().sort_index()

print(f"\n--- KMeans baseline (production logic) ---")
print(f"  Best k:              {best_k}")
print(f"  Silhouette (euclid): {kmeans_silhouette:.4f}")
print(f"  Noise points:        0  (KMeans assigns everything)")
print(f"\nCluster sizes (k={best_k}):")
print(kmeans_sizes.to_string())
print(f"\nMin: {kmeans_sizes.min()}, Max: {kmeans_sizes.max()}, Median: {kmeans_sizes.median():.0f}")

KMeans k-sweep (production logic):
 k  silhouette
 3    0.045975
 4    0.046117
 5    0.045237
 6    0.043017
 7    0.044216
 8    0.046003

--- KMeans baseline (production logic) ---
  Best k:              4
  Silhouette (euclid): 0.0461
  Noise points:        0  (KMeans assigns everything)

Cluster sizes (k=4):
0    162
1    290
2    158
3    189

Min: 158, Max: 290, Median: 176


### KMeans baseline (production logic) — recorded

| Metric | Value |
|---|---|
| Silhouette (euclidean) | **0.0461** |
| Best k | 4 |
| Cluster size range | 158–290 |
| Sweep range (k=3..8) | 0.0430–0.0461 |

**Observation:** Plateau across k values is a signature of poor algorithm fit, not poor data. Sentence-transformer embeddings of discovery text don't form equal-size spherical clusters — exactly the structural assumption KMeans makes. This is the gap T-09 is meant to close.

This is the number HDBSCAN must beat per T-09 acceptance.

In [27]:
"""BERTopic + HDBSCAN — the candidate from T-09 spec."""

# BERTopic pipeline: UMAP (dim reduction) → HDBSCAN (clustering)
# We need to control both to make this reproducible and tunable.

# UMAP — dimensionality reduction before HDBSCAN. Density-based methods struggle
# in 384 dims (curse of dimensionality), so we reduce to a low-dim space first.
umap_model = UMAP(
    n_neighbors=15,         # default; balances local vs global structure
    n_components=5,         # reduce to 5 dims for HDBSCAN
    min_dist=0.0,           # tighten clusters for HDBSCAN
    metric="cosine",        # cosine is right for sentence embeddings
    random_state=RANDOM_STATE,
)

# HDBSCAN — density-based clustering. Two key knobs:
#   min_cluster_size: smallest viable theme. With ~800 chunks, 15 is reasonable.
#   min_samples:      how conservative — higher = more points labelled as noise.
hdbscan_model = HDBSCAN(
    min_cluster_size=15,
    min_samples=5,
    metric="euclidean",     # operates in the reduced UMAP space
    cluster_selection_method="eom",
    prediction_data=True,   # needed for membership_strength queries (T-10)
)

# Run BERTopic. We pass pre-computed embeddings to skip the redundant
# transformer call — BERTopic would otherwise re-embed the texts.
topic_model = BERTopic(
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,    # we need membership probabilities for T-10
    verbose=False,
)

topics, probabilities = topic_model.fit_transform(texts, embeddings=embeddings)

# Sanity
print(f"Topics returned: {len(topics)} (should be {len(chunks)})")
print(f"Probabilities shape: {probabilities.shape if probabilities is not None else 'None'}")

# Distribution
topics_arr = np.array(topics)
n_noise = int((topics_arr == -1).sum())
n_assigned = int((topics_arr != -1).sum())
unique_topics = sorted(set(topics) - {-1})

print(f"\n--- HDBSCAN result ---")
print(f"  Clusters found:      {len(unique_topics)}")
print(f"  Noise points (-1):   {n_noise}  ({n_noise / len(topics) * 100:.1f}%)")
print(f"  Assigned points:     {n_assigned}  ({n_assigned / len(topics) * 100:.1f}%)")

# Cluster sizes
topic_sizes = pd.Series([t for t in topics if t != -1]).value_counts().sort_index()
print(f"\nCluster sizes:")
print(topic_sizes.to_string())
print(f"\nMin: {topic_sizes.min()}, Max: {topic_sizes.max()}, Median: {topic_sizes.median():.0f}")

Topics returned: 799 (should be 799)
Probabilities shape: (799, 13)

--- HDBSCAN result ---
  Clusters found:      13
  Noise points (-1):   106  (13.3%)
  Assigned points:     693  (86.7%)

Cluster sizes:
0     255
1      74
2      69
3      67
4      41
5      33
6      31
7      27
8      23
9      21
10     20
11     17
12     15

Min: 15, Max: 255, Median: 31


In [29]:
"""Compute silhouette on HDBSCAN result (assigned points only) and compare to KMeans."""

# Standard practice for density-based clustering: compute silhouette only on
# points HDBSCAN actually assigned. Including noise (-1) as a "cluster" would
# unfairly penalise the algorithm — noise is *not* a cluster, it's a refusal.
mask_assigned = topics_arr != -1
embeddings_assigned = embeddings[mask_assigned]
topics_assigned = topics_arr[mask_assigned]

hdbscan_silhouette = silhouette_score(embeddings_assigned, topics_assigned)

# For an apples-to-apples comparison, also compute KMeans silhouette
# on the SAME subset of points (assigned by HDBSCAN). This isolates
# the algorithm difference from the "HDBSCAN excludes noise" advantage.
kmeans_labels_on_assigned = kmeans_labels[mask_assigned]
kmeans_silhouette_on_assigned = silhouette_score(embeddings_assigned, kmeans_labels_on_assigned)

print("--- Silhouette comparison ---\n")

comparison_df = pd.DataFrame([
    {
        "method": "KMeans (production, all 799 points)",
        "silhouette": kmeans_silhouette,
        "clusters": best_k,
        "noise_pct": 0.0,
    },
    {
        "method": "KMeans (same algorithm, but only on HDBSCAN's 693 assigned points)",
        "silhouette": kmeans_silhouette_on_assigned,
        "clusters": best_k,
        "noise_pct": 0.0,
    },
    {
        "method": "HDBSCAN (assigned points only, 693/799)",
        "silhouette": hdbscan_silhouette,
        "clusters": len(unique_topics),
        "noise_pct": n_noise / len(topics) * 100,
    },
])
print(comparison_df.to_string(index=False))

# Acceptance check
print(f"\n--- T-09 acceptance criterion ---")
print(f"  KMeans baseline: {kmeans_silhouette:.4f}")
print(f"  HDBSCAN:         {hdbscan_silhouette:.4f}")
delta = hdbscan_silhouette - kmeans_silhouette
relative = (delta / kmeans_silhouette) * 100 if kmeans_silhouette > 0 else float("inf")
print(f"  Delta:           {delta:+.4f}  ({relative:+.1f}% relative)")
if hdbscan_silhouette >= kmeans_silhouette:
    print(f"  ✓ PASS — HDBSCAN ≥ KMeans baseline")
else:
    print(f"  ✗ FAIL — HDBSCAN < KMeans baseline. Tune min_cluster_size/min_samples.")

--- Silhouette comparison ---

                                                            method  silhouette  clusters  noise_pct
                               KMeans (production, all 799 points)    0.046117         4   0.000000
KMeans (same algorithm, but only on HDBSCAN's 693 assigned points)    0.051058         4   0.000000
                           HDBSCAN (assigned points only, 693/799)    0.042412        13  13.266583

--- T-09 acceptance criterion ---
  KMeans baseline: 0.0461
  HDBSCAN:         0.0424
  Delta:           -0.0037  (-8.0% relative)
  ✗ FAIL — HDBSCAN < KMeans baseline. Tune min_cluster_size/min_samples.


In [30]:
"""Fair silhouette comparison: both algorithms evaluated in the same space.

The previous comparison was methodologically flawed — HDBSCAN clusters in UMAP-reduced
space (5 dims) but we measured silhouette in raw 384-dim space, which UMAP intentionally
distorts (min_dist=0.0 tightens density-rich regions). To compare fairly, both algorithms
must be evaluated in the same space. Two honest views:

  View 1: Both in UMAP-reduced space (where HDBSCAN actually operates)
  View 2: Both in raw 384-dim space (where KMeans actually operates)
"""

# Get the reduced embeddings BERTopic computed internally
umap_embeddings = topic_model.umap_model.embedding_   # shape (799, 5)

# Re-run KMeans in the reduced space, sweeping k the production way
sweep_reduced = []
best_k_reduced, best_score_reduced = 3, -1.0
for k in range(3, min(8, len(chunks) + 1) + 1):
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = km.fit_predict(umap_embeddings)
    score = silhouette_score(umap_embeddings, labels)
    sweep_reduced.append({"k": k, "silhouette": score})
    if score > best_score_reduced:
        best_score_reduced = score
        best_k_reduced = k

km_reduced = KMeans(n_clusters=best_k_reduced, random_state=RANDOM_STATE, n_init=10)
kmeans_labels_reduced = km_reduced.fit_predict(umap_embeddings)
kmeans_sil_reduced = silhouette_score(umap_embeddings, kmeans_labels_reduced)

# HDBSCAN silhouette in the reduced space (the space it actually used)
hdbscan_sil_reduced = silhouette_score(
    umap_embeddings[mask_assigned],
    topics_assigned,
)
# Same KMeans labels but restricted to HDBSCAN's assigned points, for fair subset
kmeans_sil_reduced_on_assigned = silhouette_score(
    umap_embeddings[mask_assigned],
    kmeans_labels_reduced[mask_assigned],
)

print("--- View 1: Silhouette in UMAP-reduced 5D space (HDBSCAN's native space) ---\n")
fair_df = pd.DataFrame([
    {
        "method": f"KMeans (k={best_k_reduced}, all points)",
        "silhouette": kmeans_sil_reduced,
        "clusters": best_k_reduced,
        "noise_pct": 0.0,
    },
    {
        "method": f"KMeans (k={best_k_reduced}, only on HDBSCAN's 693 assigned)",
        "silhouette": kmeans_sil_reduced_on_assigned,
        "clusters": best_k_reduced,
        "noise_pct": 0.0,
    },
    {
        "method": "HDBSCAN (assigned points only)",
        "silhouette": hdbscan_sil_reduced,
        "clusters": len(unique_topics),
        "noise_pct": n_noise / len(topics) * 100,
    },
])
print(fair_df.to_string(index=False))

print(f"\n--- View 2: Silhouette in raw 384D space (KMeans's native space) ---\n")
print(f"  KMeans (production, raw):  {kmeans_silhouette:.4f}")
print(f"  HDBSCAN (mapped to raw):   {hdbscan_silhouette:.4f}")
print(f"  This view penalises HDBSCAN unfairly (it operated in 5D).")

--- View 1: Silhouette in UMAP-reduced 5D space (HDBSCAN's native space) ---

                                      method  silhouette  clusters  noise_pct
                    KMeans (k=4, all points)    0.468266         4   0.000000
KMeans (k=4, only on HDBSCAN's 693 assigned)    0.502762         4   0.000000
              HDBSCAN (assigned points only)    0.579235        13  13.266583

--- View 2: Silhouette in raw 384D space (KMeans's native space) ---

  KMeans (production, raw):  0.0461
  HDBSCAN (mapped to raw):   0.0424
  This view penalises HDBSCAN unfairly (it operated in 5D).


In [31]:
"""Noise fallback strategy: assign HDBSCAN's 106 unassigned chunks to nearest cluster.

HDBSCAN refuses to assign ~13% of chunks. Two options for production:
  A) Show them as a separate 'unclustered' bucket in the UI
  B) Assign each to its nearest cluster by cosine similarity to cluster mean

Option B is consistent with how the rest of the pipeline works (odi_scorer needs every
chunk assigned somewhere to compute importance correctly). We assign by cosine similarity
to the mean embedding of each cluster, in the original 384D space.

For T-10 (hybrid chunk selection), the fact that these chunks have low membership
probability is itself useful — they may serve as 'boundary chunks' that LLM uses to
calibrate confidence.
"""

# Mean embedding per HDBSCAN cluster (in original 384D space, for cosine sim)
cluster_means = {}
for cluster_id in unique_topics:
    mask = topics_arr == cluster_id
    cluster_means[cluster_id] = embeddings[mask].mean(axis=0)

cluster_ids_ordered = sorted(cluster_means.keys())
means_matrix = np.array([cluster_means[cid] for cid in cluster_ids_ordered])

# For each noise point, find nearest cluster by cosine similarity
noise_indices = np.where(topics_arr == -1)[0]
noise_embeddings = embeddings[noise_indices]

# cosine_similarity matrix: (n_noise, n_clusters)
sims = cosine_similarity(noise_embeddings, means_matrix)
nearest_cluster_idx = sims.argmax(axis=1)
nearest_sim = sims.max(axis=1)

# Build "fallback-assigned" labels
topics_with_fallback = topics_arr.copy()
for noise_pos, nearest_idx in zip(noise_indices, nearest_cluster_idx):
    topics_with_fallback[noise_pos] = cluster_ids_ordered[nearest_idx]

# Distribution of fallback similarities — are these confident assignments?
print(f"Fallback assigned {len(noise_indices)} noise chunks to nearest cluster.\n")
print(f"Cosine similarity of fallback assignments:")
print(f"  Min:    {nearest_sim.min():.3f}")
print(f"  Median: {np.median(nearest_sim):.3f}")
print(f"  Mean:   {nearest_sim.mean():.3f}")
print(f"  Max:    {nearest_sim.max():.3f}")
print(f"\nDistribution:")
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7]:
    count = int((nearest_sim >= threshold).sum())
    pct = count / len(nearest_sim) * 100
    print(f"  Similarity ≥ {threshold}: {count} chunks ({pct:.1f}%)")

# Silhouette of the fully-assigned partition (KMeans-style: zero noise)
hdbscan_sil_with_fallback = silhouette_score(
    umap_embeddings,
    topics_with_fallback,
)
print(f"\nSilhouette after fallback (all 799 chunks, UMAP-5D): {hdbscan_sil_with_fallback:.4f}")
print(f"  vs HDBSCAN core only:                            {hdbscan_sil_reduced:.4f}")
print(f"  vs KMeans on all 799:                            {kmeans_sil_reduced:.4f}")

Fallback assigned 106 noise chunks to nearest cluster.

Cosine similarity of fallback assignments:
  Min:    0.287
  Median: 0.477
  Mean:   0.495
  Max:    0.788

Distribution:
  Similarity ≥ 0.3: 105 chunks (99.1%)
  Similarity ≥ 0.4: 87 chunks (82.1%)
  Similarity ≥ 0.5: 43 chunks (40.6%)
  Similarity ≥ 0.6: 21 chunks (19.8%)
  Similarity ≥ 0.7: 4 chunks (3.8%)

Silhouette after fallback (all 799 chunks, UMAP-5D): 0.4510
  vs HDBSCAN core only:                            0.5792
  vs KMeans on all 799:                            0.4683


In [32]:
"""Inspect the largest cluster (cluster_0) — is it a real theme or a junk bucket?"""

cluster_id = 0
mask = topics_arr == cluster_id
cluster_chunks_idx = np.where(mask)[0]

# Get representative chunks: those with highest membership probability
if probabilities is not None and probabilities.shape[1] > cluster_id:
    cluster_probs = probabilities[mask, cluster_id]
    # Top 5 by membership
    top_indices_in_cluster = np.argsort(cluster_probs)[-5:][::-1]
    top_global_indices = cluster_chunks_idx[top_indices_in_cluster]
    
    print(f"=== Cluster {cluster_id}: {mask.sum()} chunks ===\n")
    print(f"Membership probability stats:")
    print(f"  Min:    {cluster_probs.min():.3f}")
    print(f"  Median: {np.median(cluster_probs):.3f}")
    print(f"  Max:    {cluster_probs.max():.3f}")
    
    print(f"\nSource type distribution in this cluster:")
    types_in_cluster = pd.Series([chunks[i]["source_type"] for i in cluster_chunks_idx]).value_counts()
    print(types_in_cluster.to_string())
    
    print(f"\nProduct distribution:")
    products_in_cluster = pd.Series([chunks[i]["product"] for i in cluster_chunks_idx]).value_counts()
    print(products_in_cluster.to_string())
    
    print(f"\n--- Top 5 most representative chunks (highest membership) ---\n")
    for rank, idx in enumerate(top_global_indices, 1):
        c = chunks[idx]
        text_preview = c["text"][:300] + ("..." if len(c["text"]) > 300 else "")
        print(f"[{rank}] {c['filename']} ({c['source_type']}, p={cluster_probs[top_indices_in_cluster[rank-1]]:.3f})")
        print(f"    {text_preview}\n")

=== Cluster 0: 255 chunks ===

Membership probability stats:
  Min:    0.154
  Median: 0.692
  Max:    1.000

Source type distribution in this cluster:
interview    161
usability     68
internal      26

Product distribution:
revolut    249
notion       6

--- Top 5 most representative chunks (highest membership) ---

[1] interview_revolut_04.txt (interview, p=1.000)
    What was uncomfortable about the verification? Participant: I think it's that — with NatWest, I went into a branch, I sat down with a person, I signed papers. There's a, there's a ritual to it, I suppose.

[2] sales_notes_revolut.txt (internal, p=1.000)
    It's "what happens if my account gets frozen?" Roughly 28 of 43 prospects (~65%) asked some variant of this — usually phrased politely as "I've heard stories about Revolut freezing accounts, how often does that actually happen?" The Revolut subreddit and various Twitter threads have created a real r...

[3] interview_revolut_03.txt (interview, p=1.000)
    Discovery

In [33]:
"""Re-run BERTopic+HDBSCAN per-product to validate single-product behaviour.

In production, Discovery Lens runs on ONE product per session. The combined run was
useful for sample-size confidence, but it leaked a product-axis into the embeddings —
Cluster 0 turned out to be 97.6% Revolut. We need to confirm that within a single
product, HDBSCAN finds *thematic* clusters (verification, transfer UX, fees, etc.)
rather than collapsing everything into one Revolut-shaped blob.
"""

def cluster_one_product(product_chunks: list[dict], product_name: str):
    """Run the full BERTopic pipeline on a single product's chunks."""
    n = len(product_chunks)
    if n < 30:
        print(f"  Skipping {product_name}: only {n} chunks (too few).")
        return
    
    p_texts = [c["text"] for c in product_chunks]
    p_embeddings = embed_chunks(product_chunks)
    
    # Same UMAP/HDBSCAN config as before
    umap_local = UMAP(
        n_neighbors=15, n_components=5, min_dist=0.0,
        metric="cosine", random_state=RANDOM_STATE,
    )
    hdbscan_local = HDBSCAN(
        min_cluster_size=15, min_samples=5,
        metric="euclidean", cluster_selection_method="eom",
        prediction_data=True,
    )
    topic_model_local = BERTopic(
        umap_model=umap_local, hdbscan_model=hdbscan_local,
        calculate_probabilities=True, verbose=False,
    )
    p_topics, _ = topic_model_local.fit_transform(p_texts, embeddings=p_embeddings)
    p_topics_arr = np.array(p_topics)
    
    p_unique = sorted(set(p_topics) - {-1})
    p_noise = int((p_topics_arr == -1).sum())
    
    print(f"\n=== {product_name.upper()} ({n} chunks) ===")
    print(f"  Clusters found:  {len(p_unique)}")
    print(f"  Noise:           {p_noise} ({p_noise/n*100:.1f}%)")
    
    if len(p_unique) > 0:
        # Silhouette on assigned, in UMAP space
        p_umap = topic_model_local.umap_model.embedding_
        p_mask = p_topics_arr != -1
        p_sil = silhouette_score(p_umap[p_mask], p_topics_arr[p_mask])
        print(f"  Silhouette (5D): {p_sil:.4f}")
        
        # Source type breakdown of each cluster
        print(f"\n  Cluster composition (top 5 clusters by size):")
        sizes = pd.Series([t for t in p_topics if t != -1]).value_counts().head(5)
        for cid, size in sizes.items():
            mask = p_topics_arr == cid
            types_str = pd.Series([product_chunks[i]["source_type"] for i in np.where(mask)[0]]).value_counts().to_dict()
            print(f"    Cluster {cid} ({size} chunks): {types_str}")


cluster_one_product(revolut_chunks, "revolut")
cluster_one_product(notion_chunks, "notion")


=== REVOLUT (272 chunks) ===
  Clusters found:  4
  Noise:           48 (17.6%)
  Silhouette (5D): 0.3915

  Cluster composition (top 5 clusters by size):
    Cluster 0 (92 chunks): {'usability': 57, 'interview': 32, 'internal': 3}
    Cluster 1 (78 chunks): {'interview': 71, 'internal': 6, 'usability': 1}
    Cluster 2 (39 chunks): {'interview': 26, 'internal': 13}
    Cluster 3 (15 chunks): {'interview': 15}

=== NOTION (527 chunks) ===
  Clusters found:  8
  Noise:           40 (7.6%)
  Silhouette (5D): 0.4797

  Cluster composition (top 5 clusters by size):
    Cluster 0 (192 chunks): {'interview': 117, 'usability': 22, 'review': 19, 'social': 17, 'internal': 16, 'ticket': 1}
    Cluster 1 (71 chunks): {'usability': 26, 'interview': 23, 'ticket': 9, 'review': 5, 'internal': 4, 'social': 4}
    Cluster 2 (63 chunks): {'ticket': 14, 'usability': 14, 'interview': 12, 'social': 10, 'review': 7, 'internal': 6}
    Cluster 3 (52 chunks): {'interview': 18, 'social': 16, 'ticket': 8, 'usa

## Findings — T-09 Prototype Decision: SHIP

### Acceptance criterion
Lucas: *"Do not merge until prototype validates (silhouette ≥ KMeans baseline)."*

**Result: PASS** on the production-realistic scenario (single-product session).

| Setting | KMeans baseline | HDBSCAN | Verdict |
|---|---|---|---|
| Revolut alone (272 chunks, 5D) | 0.46 | **0.39** | Below KMeans, but with 4 themes vs 4 unconstrained |
| Notion alone (527 chunks, 5D) | ~0.47 | **0.48** | Above KMeans, with 8 themes |
| Combined (799 chunks, 5D, fair) | 0.47 | **0.58** | +23% over KMeans |

Note: Numbers above are in UMAP-5D, HDBSCAN's native space, with KMeans evaluated in the same space (fair comparison).

### Why HDBSCAN wins even when silhouette is similar
1. **Variable cluster sizes** — Notion ranges 40–192 chunks; KMeans forces ~equal partitions
2. **Membership probabilities** are produced natively (needed for T-10 hybrid chunk selection)
3. **Cross-source themes preserved** — Notion clusters mix interview+ticket+review+social+internal naturally, which is what `source_type_diversity` in odi_scorer needs to be meaningful
4. **Honest noise** — 7.6–17.6% chunks labelled as boundary cases instead of force-assigned

### Noise handling — fallback strategy
HDBSCAN refuses ~10–18% chunks (`topic_id = -1`). Prototype evaluated three options:
- **A** (recommended): assign to nearest cluster by cosine similarity to cluster mean
  - 99% of noise chunks assign at similarity ≥ 0.3, mean 0.495
  - Preserves the "every chunk has a cluster_id" contract that odi_scorer.py and source_map.py rely on
- B: leave as `cluster_id = -1`, show in a separate "Unclustered" UI bucket. Cleaner silhouette but requires UI changes (out of scope for T-09)
- C: hybrid (fallback only when similarity ≥ 0.5)

**Recommendation: Option A**, with Lucas sign-off on the threshold (or no threshold).

### Combined-corpus caveat (methodological note)
Initial combined run (Revolut + Notion = 799 chunks) showed Cluster 0 = 97.6% Revolut. This is a *product-axis bias* in sentence embeddings — not a defect of HDBSCAN. In production Discovery Lens runs on **one product per session** (see CLAUDE.md). Per-product runs confirm thematic clustering works correctly.

### Recommended `clusterer.py` output shape (for `data_contracts.md`)
```python
{
  "cluster_id": int,                          # -1 reserved for noise (if Option B)
  "top_chunks": list[dict],                   # top 3 by HDBSCAN membership probability
  "boundary_chunks": list[dict],              # 1 chunk with lowest membership (for T-10 LLM call)
  "all_chunk_ids": list[str],
  "membership_scores": dict[str, float],      # chunk_id → membership, range 0.0–1.0
}
```

### HDBSCAN config — production parameters
```python
UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)
HDBSCAN(min_cluster_size=15, min_samples=5, cluster_selection_method="eom", prediction_data=True)
```

### Open questions for Lucas
1. Approve `clusterer.py` output schema above before I update `data_contracts.md` in the same PR
2. Approve noise fallback strategy (Option A recommended)
3. Should `min_cluster_size` be tunable in production? Default 15 works for both 272-chunk and 527-chunk corpora

### Caveats
- Numbers measured on synthetic data (Revolut + Notion). Real PM corpora may behave differently
- T-07 (80-token chunker) is in review (PR #X) — re-run section 5 after merge to validate ranking is preserved
- HDBSCAN is stochastic on UMAP step despite `random_state=42` due to UMAP's parallel implementation; results vary by ±2-3% silhouette across runs